# Nonlinear static analysis with structsolve

The plate of the notebook *Nonlinear static analysis*, solved with
`structsolve.Analysis` of [structsolve](https://saullocastro.github.io/structsolve/),
see the section *Solving with structsolve* of the user guide. The four
callables loop over the elements with the functions of the notebook
*Nonlinear static analysis* and work on the unknown degrees of freedom `bu`,
the partition of the free degrees of freedom; `calc_fint` also keeps the
internal forces of each iteration. The code is that of the unit test
`tests/test_static_axial_load_nonlinear.py`.

## Theory

`structsolve.Analysis` solves the equilibrium
$\boldsymbol{R}(\boldsymbol{u}, \lambda) =
\boldsymbol{F}_{int}(\boldsymbol{u}) - \lambda \boldsymbol{F}_{ext} =
\boldsymbol{0}$ with the full Newton-Raphson method and the tangent
stiffness matrix

$$
\boldsymbol{K}_T = \boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL}(\boldsymbol{u})
+ \boldsymbol{K}_{GNL}(\boldsymbol{u}) + \boldsymbol{K}_G(\boldsymbol{u})
$$

given by the callables: `calc_kC` returns $\boldsymbol{K}_{C0}$, or
$\boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} + \boldsymbol{K}_{GNL}$ with
`NLgeom=True`, and `calc_kG` returns $\boldsymbol{K}_G$, such that
`calc_kC + calc_kG` is $\boldsymbol{K}_T$. The convergence criterion is, by
default, Crisfield's: the residual and the reference forces are scaled with
the diagonal of the linear stiffness matrix over the free degrees of
freedom, and the iterations stop when their ratio is below
`Analysis.relTOL = 1e-9`. The scaled norm does not depend on the units of
the degrees of freedom, which mix displacements, slopes and $w_{,xy}$. A
tangent stiffness matrix that is the exact Jacobian of the internal forces
squares this residual at the last iteration.

The imports and the module-level definitions of the test, with the helper
functions that the code below uses:

In [1]:
import numpy as np
from numpy import isclose
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh, spsolve

from composites import isotropic_plate
from structsolve import Analysis
from structsolve.convergence import crisfield_weights, crisfield_zeta

from bfscplate2d import (BFSCPlate2D, update_KC0, update_KCNL, update_KG,
        update_fint, DOF, DOUBLE, INT, KC0_SPARSE_SIZE, KCNL_SPARSE_SIZE,
        KG_SPARSE_SIZE)
from bfscplate2d.quadrature import get_points_weights

# number of nodes
nx = 13
ny = 7

# geometry
a = 0.9
b = 0.5

# material properties
E = 0.7e11
nu = 0.3
h = 0.001


class Model(object):
    """Mesh, elements, boundary conditions and the operators built on them."""


def build_model():
    points, weights = get_points_weights(nint=4)
    lam = isotropic_plate(thickness=h, E=E, nu=nu)

    xlin = np.linspace(0, a, nx)
    ylin = np.linspace(0, b, ny)
    xmesh, ymesh = np.meshgrid(xlin, ylin)

    # getting nodes
    ncoords = np.vstack((xmesh.T.flatten(), ymesh.T.flatten())).T
    x = ncoords[:, 0]
    y = ncoords[:, 1]
    nids = 1 + np.arange(ncoords.shape[0])
    nid_pos = dict(zip(nids, np.arange(len(nids))))

    nids_mesh = nids.reshape(nx, ny)

    n1s = nids_mesh[:-1, :-1].flatten()
    n2s = nids_mesh[1:, :-1].flatten()
    n3s = nids_mesh[1:, 1:].flatten()
    n4s = nids_mesh[:-1, 1:].flatten()

    num_elements = len(n1s)
    print('num_elements', num_elements)

    N = DOF*nx*ny
    KC0r = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    KC0c = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    KC0v = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    KCNLr = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
    KCNLc = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
    KCNLv = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    KGr = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
    KGc = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
    KGv = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    init_k_KC0 = 0
    init_k_KCNL = 0
    init_k_KG = 0

    elements = []
    for n1, n2, n3, n4 in zip(n1s, n2s, n3s, n4s):
        plate = BFSCPlate2D()
        plate.n1 = n1
        plate.n2 = n2
        plate.n3 = n3
        plate.n4 = n4
        plate.c1 = DOF*nid_pos[n1]
        plate.c2 = DOF*nid_pos[n2]
        plate.c3 = DOF*nid_pos[n3]
        plate.c4 = DOF*nid_pos[n4]
        plate.ABD = lam.ABD
        plate.lex = a/(nx - 1)
        plate.ley = b/(ny - 1)
        plate.init_k_KC0 = init_k_KC0
        plate.init_k_KCNL = init_k_KCNL
        plate.init_k_KG = init_k_KG
        update_KC0(plate, points, weights, KC0r, KC0c, KC0v)
        init_k_KC0 += KC0_SPARSE_SIZE
        init_k_KCNL += KCNL_SPARSE_SIZE
        init_k_KG += KG_SPARSE_SIZE
        elements.append(plate)

    KC0 = coo_matrix((KC0v, (KC0r, KC0c)), shape=(N, N)).tocsc()

    # applying boundary conditions
    # simply supported
    bk = np.zeros(N, dtype=bool)
    checkSS = isclose(x, 0) | isclose(x, a)
    bk[3::DOF] = checkSS
    bk[6::DOF] = checkSS
    checkSS = isclose(y, 0) | isclose(y, b)
    bk[6::DOF] += checkSS
    check = isclose(x, a/2.)
    bk[0::DOF] = check
    bu = ~bk
    print('boundary conditions bu', bu.sum())

    m = Model()
    m.points, m.weights, m.lam = points, weights, lam
    m.ncoords, m.x, m.y = ncoords, x, y
    m.xmesh, m.ymesh = xmesh, ymesh
    m.nid_pos = nid_pos
    m.elements = elements
    m.N = N
    m.KC0 = KC0
    m.KC0uu = KC0[bu, :][:, bu]
    m.bk, m.bu = bk, bu
    m.KCNLr, m.KCNLc, m.KCNLv = KCNLr, KCNLc, KCNLv
    m.KGr, m.KGc, m.KGv = KGr, KGc, KGv
    return m


def calc_KCNL(m, u):
    """The nonlinear part KCNL(u) of the tangent, K_CNL + K_GNL."""
    m.KCNLv *= 0
    for plate in m.elements:
        update_KCNL(u, plate, m.points, m.weights, m.KCNLr, m.KCNLc, m.KCNLv)
    return coo_matrix((m.KCNLv, (m.KCNLr, m.KCNLc)),
                      shape=(m.N, m.N)).tocsc()


def calc_KG(m, u):
    """The geometric stiffness KG(u), of the tangent and of the linearised
    comparisons."""
    m.KGv *= 0
    for plate in m.elements:
        update_KG(u, plate, m.points, m.weights, m.KGr, m.KGc, m.KGv)
    return coo_matrix((m.KGv, (m.KGr, m.KGc)), shape=(m.N, m.N)).tocsc()


def calc_fint(m, u):
    fint = np.zeros(m.N)
    for plate in m.elements:
        update_fint(u, plate, m.points, m.weights, fint)
    return fint


def forces(m, load, transverse):
    """Axial compression along x, plus a transverse load at the centre."""
    fext = np.zeros(m.N)
    checkBottomEdge = isclose(m.x, 0)
    checkTopEdge = isclose(m.x, a)
    fext[0::DOF][checkBottomEdge] = +load/ny
    assert isclose(fext.sum(), load)
    fext[0::DOF][checkTopEdge] = -load/ny
    assert isclose(fext.sum(), 0)

    ftrans = np.zeros(m.N)
    check = isclose(m.x, a/2) & isclose(m.y, b/2)
    assert check.sum() == 1
    ftrans[6::DOF][check] = transverse
    return fext, ftrans


def linear_buckling_load(m, load):
    """The first buckling load, for the reference amplification."""
    fext, _ = forces(m, load, 0.)
    u0 = np.zeros(m.N)
    u0[m.bu] = spsolve(m.KC0uu, fext[m.bu])
    KGuu = calc_KG(m, u0)[m.bu, :][:, m.bu]
    eigvals, eigvecsu = eigsh(A=KGuu, k=5, which='SM', M=m.KC0uu, tol=1e-6,
                              sigma=1., mode='cayley')
    eigvals = -1./eigvals
    return eigvals[0]*load, u0

In [2]:
class Callables(object):
    """The callables of structsolve.Analysis for the model ``m`` under the
    external forces ``fext``.

    They loop over the elements with the update functions and assemble, and
    work on the unknown degrees of freedom only, the partition of the free
    dofs of structsolve: the solution vector of the solvers is ``c =
    u[bu]``, and the force vectors and matrices are restricted to ``bu``.
    ``calc_fint`` also keeps the load factor and the internal forces of each
    iteration, for the residual history.
    """
    def __init__(self, m, fext):
        self.m = m
        self.fext = fext
        self.iterations = []

    def expand(self, c):
        """The displacements of the model from those of the unknown dofs."""
        u = np.zeros(self.m.N)
        u[self.m.bu] = c
        return u

    def calc_fext(self, inc=1., silent=False):
        return inc*self.fext[self.m.bu]

    def calc_fint(self, c, inc=1., silent=False):
        fint = calc_fint(self.m, self.expand(c))[self.m.bu]
        self.iterations.append((inc, fint))
        return fint

    def calc_kC(self, c=None, NLgeom=False, silent=False):
        if not NLgeom:
            return self.m.KC0uu
        bu = self.m.bu
        KCNL = calc_KCNL(self.m, self.expand(c))
        return (self.m.KC0 + KCNL)[bu, :][:, bu]

    def calc_kG(self, c=None, NLgeom=False, silent=False):
        bu = self.m.bu
        return calc_KG(self.m, self.expand(c))[bu, :][:, bu]

The analysis applies the load in equal increments, with
`Analysis.initialInc` and `Analysis.maxInc`, checks that none was cut, and
computes Crisfield's relative residual, the convergence criterion of
structsolve, at each iteration of the last increment:

In [3]:
def newton_raphson(m, fext, increments=1):
    """Full Newton-Raphson of structsolve.Analysis, with ``increments``
    equal load increments. Returns the solution and the residual history of
    the last increment.

    The tangent is rebuilt every iteration, so the residual history is the
    direct evidence of whether KT is the derivative of fint: a consistent
    tangent squares the residual each step, an inconsistent one multiplies
    it by a constant factor instead. The residual is measured with
    Crisfield's criterion, the convergence criterion of structsolve, with
    its default tolerance of 1e-9.
    """
    callables = Callables(m, fext)
    an = Analysis(calc_fext=callables.calc_fext,
                  calc_fint=callables.calc_fint,
                  calc_kC=callables.calc_kC,
                  calc_kG=callables.calc_kG)
    an.initialInc = 1./increments
    an.maxInc = 1./increments
    increments_done, cs = an.static(NLgeom=True)
    # every increment converged, none was cut
    assert np.allclose(increments_done,
                       np.linspace(0, 1, increments + 1)[1:]), (
            'Not converged! load factors %s' % increments_done)
    u = callables.expand(cs[-1])

    w = crisfield_weights(m.KC0uu)
    f = fext[m.bu]
    history = [crisfield_zeta(inc*f - fint, inc*f, fint, w)
               for inc, fint in callables.iterations if inc == 1.]
    return u, history

A consistent tangent stiffness matrix squares the residual at the last iteration:

In [4]:
def assert_quadratic_convergence(history, epsilon=1.e-9):
    """The last Newton step must square the residual, not merely shrink it.

    Quadratic convergence is an asymptotic statement, so it is the final
    transition that carries it: the step that takes the iterate from above
    the tolerance to below it. Earlier steps are not a fair test here,
    because the residual is normalised by the whole external load, which the
    axial part may satisfy at the first step while the out-of-plane part is
    still far from converged.

    With an inconsistent tangent the residual falls by a roughly constant
    factor throughout, so no step ever squares it and this fails.
    """
    assert history[-1] < epsilon, 'did not reach the tolerance'
    assert len(history) <= 6, (
        'took %d iterations to converge; a consistent tangent needs a '
        'handful' % len(history))
    assert len(history) >= 2, 'no Newton step was taken'
    prev, curr = history[-2], history[-1]
    # the floor keeps the test meaningful once prev**2 drops below the
    # round-off level of the residual itself, about 1e-14 here: the last
    # increment of test_large_deflection goes 2.7e-8 -> 1.5e-14
    assert curr < max(10.*prev**2, 1.e-13), (
        'the last step took the residual %.3e -> %.3e, which is linear '
        'rather than quadratic; the tangent is not the derivative of fint'
        % (prev, curr))

With a small transverse force the solution in one increment reproduces the
linearised one, $(\boldsymbol{K}_{C0} + \boldsymbol{K}_G)^{-1}\boldsymbol{F}_{ext}$,
with $\boldsymbol{K}_G$ of the axial load:

The test checks that the nonlinear solution reproduces the linearised one,
$(\boldsymbol{K}_{C0} + \boldsymbol{K}_G(P))^{-1}\boldsymbol{F}_t$, within
$10^{-3}$, since $w_{max}/h \approx 0.011$ makes the membrane stretching
negligible; that the amplification of the deflection over the response to
the transverse force alone, $\boldsymbol{K}_{C0}^{-1}\boldsymbol{F}_t$, is
between 1 and the single-mode ceiling $1/(1 - P/P_{cr})$, with $P_{cr}$ the
first buckling load, since a point load excites many modes; and $P_{cr}$
and the maximum deflection against the reference values stored in the test,
with a relative tolerance of $10^{-3}$.

In [5]:
def test_static(plot=False):
    """Small transverse load: the linearised buckling amplification."""
    load = 300.  # N
    transverse = 0.01  # N

    m = build_model()
    fext, ftrans = forces(m, load, transverse)

    load_cr, u_axial = linear_buckling_load(m, load)
    print('load_cr', load_cr)
    assert isclose(load_cr, 501.59022126865693, rtol=1e-3)
    assert load < load_cr

    # the exact linearised answer: the transverse load resisted by the
    # bending stiffness reduced by the geometric stiffness of the axial load
    KG = calc_KG(m, u_axial)
    u_lin = np.zeros(m.N)
    u_lin[m.bu] = spsolve((m.KC0 + KG)[m.bu, :][:, m.bu], ftrans[m.bu])
    w_lin = u_lin[6::DOF].max()

    # and the transverse load on its own, for the amplification factor
    u_bend = np.zeros(m.N)
    u_bend[m.bu] = spsolve(m.KC0uu, ftrans[m.bu])
    w_bend = u_bend[6::DOF].max()

    u, history = newton_raphson(m, fext + ftrans)
    assert_quadratic_convergence(history)

    w = u[6::DOF].reshape(nx, ny).T
    print('w min max', w.min(), w.max())
    print('w linearised            ', w_lin)
    print('amplification           ', w.max()/w_bend)
    print('amplification ceiling   ', 1./(1. - load/load_cr))

    if plot:
        import matplotlib
        matplotlib.use('TkAgg')
        import matplotlib.pyplot as plt
        plt.gca().set_aspect('equal')
        levels = np.linspace(w.min(), w.max(), 300)
        plt.contourf(m.xmesh, m.ymesh, w, levels=levels)
        plt.colorbar()
        plt.show()

    # w/h is about 0.011 here, so the membrane stretching is negligible and
    # the nonlinear answer must reproduce the linearised one
    assert isclose(w.max(), w_lin, rtol=1e-3)

    # a point load excites many modes, so the amplification stays below the
    # single-mode ceiling; it must still exceed one
    amplification = w.max()/w_bend
    assert 1. < amplification < 1./(1. - load/load_cr)

    assert isclose(w.max(), 1.1104539457685102e-05, rtol=1e-3)

Running the analysis:

In [6]:
test_static()

num_elements 72
boundary conditions bu 853
load_cr 501.5902212686583
Started Non-Linear Static Analysis
	___________________________________________
	                                           
	Newton-Raphson solver
	___________________________________________
	Initializing...


	Step 1, attempting load factor 1.00000
		Iteration: 1, max(|R|) = 3.145e-03, zeta = 4.028e-04


		Iteration: 2, max(|R|) = 2.297e-03, zeta = 4.095e-05


		Iteration: 3, max(|R|) = 1.739e-11, zeta = 2.802e-12
		Converged at load factor 1.00000
Finished Non-Linear Static Analysis
	    load factor 1.00000
w min max 0.0 1.110492517385695e-05
w linearised             1.1105252940206308e-05
amplification            1.761624869645655
amplification ceiling    2.488167422566551


$P_{cr} = 501.59$ N for $P = 300$ N: the amplification, 1.76, is below the
ceiling $1/(1 - P/P_{cr}) = 2.49$, and the nonlinear deflection agrees with
the linearised one to $3 \times 10^{-5}$.

## Follower pressure

The callables of a follower pressure, `FollowerCallables`, which receive the
load factor `inc`, are given in the section *Solving with structsolve* of
the user guide. The cantilever of the notebook *Follower pressure*, with
free edges and an unsymmetric load stiffness matrix, solved with these
callables in five load increments, against the reference values of panels.
The code is that of the unit test
`tests/test_follower_pressure_structsolve.py`, with the definitions it
imports from `tests/test_follower_pressure.py`.

### Theory

A follower pressure stays normal to the deformed mid-surface and acts on its
deformed area [1, 2]. Its force vector
$\boldsymbol{F}_p(\boldsymbol{u})$ depends on the displacements, and its
derivative, the load stiffness matrix
$\boldsymbol{K}_F = -\partial \boldsymbol{F}_p/\partial \boldsymbol{u}$,
enters the tangent stiffness matrix. With the load factor $\lambda$,
`calc_fext` gives $\lambda \boldsymbol{F}_p(\boldsymbol{0})$, `calc_fint`
adds $-\lambda (\boldsymbol{F}_p(\boldsymbol{u}) -
\boldsymbol{F}_p(\boldsymbol{0}))$ to the internal forces and `calc_kC` adds
the load stiffness matrix of $\lambda p$, such that the residual of
structsolve is

$$
\boldsymbol{R} = \boldsymbol{F}_{int}(\boldsymbol{u}) - \lambda
\boldsymbol{F}_p(\boldsymbol{u})
$$

and its Jacobian $\boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} +
\boldsymbol{K}_{GNL} + \boldsymbol{K}_G + \boldsymbol{K}_F$, unsymmetric for
the free edges of the cantilever and solved with LU factorisations.

The tip deflection $w$ and the tip displacement $u$ are checked within
$10^{-3}$ against the reference values `PANELS` of the plate_clpt_donnell
model of panels, the same von Kármán strains and area vector discretized
with $m = n = 16$ Bardell functions [3].

The imports and the module-level definitions of the test, with the helper
functions that the code below uses:

In [7]:
import numpy as np
from numpy import isclose
from numpy.polynomial import Polynomial
from numpy.polynomial.legendre import leggauss
import pytest
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve
from composites import isotropic_plate, laminated_plate

from bfscplate2d import (BFSCPlate2D, update_KC0, update_KCNL, update_KG,
        update_fint, update_fext_follower, update_KCfollower, DOF, DOUBLE,
        INT, KC0_SPARSE_SIZE, KCNL_SPARSE_SIZE, KG_SPARSE_SIZE,
        KCfollower_SPARSE_SIZE)
from bfscplate2d.quadrature import get_points_weights

# exact for the complete area vector with a pressure of degree 2, see the
# Notes of update_fext_follower
NINT = 6
TRUNCATIONS = [0, 1]

LAMINAPROP = (142.5e9, 142.5e9/20, 0.25, 0.5*142.5e9/20, 0.5*142.5e9/20,
              0.5*142.5e9/20)
# unsymmetric and unbalanced, such that A16, A26 and B are non-zero
LAMINATE = laminated_plate(stack=[30, -45, 0, 90], laminaprop=LAMINAPROP,
                           plyt=0.125e-3)

A, B = 0.3, 0.2


# Independent interpolation: the cubic Hermite polynomials on t in [0, 1],
# t = (xi + 1)/2, written without the functions of the element

H00 = Polynomial([1, 0, -3, 2])
H10 = Polynomial([0, 1, -2, 1])
H01 = Polynomial([0, 0, 3, -2])
H11 = Polynomial([0, 0, -1, 1])
# node 1 (xi, eta) = (-1, -1), node 2 (+1, -1), node 3 (+1, +1), node 4 (-1, +1)
CORNERS = [(0, 0), (1, 0), (1, 1), (0, 1)]


# 5, large deflection of a cantilever plate against panels

# reference values of the plate_clpt_donnell model of panels, w/h and u at
# (a, b/2) of the cantilever of test_cantilever_large_deflection_against_panels
PANELS = {
    'dead': dict(w=20.05040761, u=-3.06710517e-03),
    0: dict(w=20.23571166, u=-3.12566486e-03),
    1: dict(w=20.23571209, u=-3.12566500e-03),
}


class Mesh(object):
    """Rectangular mesh of ``nx`` by ``ny`` nodes over ``a`` by ``b``"""
    def __init__(self, nx=4, ny=3, a=A, b=B, lam=LAMINATE, nint=NINT):
        self.a, self.b = a, b
        self.points, self.weights = get_points_weights(nint=nint)
        xlin = np.linspace(0, a, nx)
        ylin = np.linspace(0, b, ny)
        xmesh, ymesh = np.meshgrid(xlin, ylin)
        ncoords = np.vstack((xmesh.T.flatten(), ymesh.T.flatten())).T
        self.x = ncoords[:, 0]
        self.y = ncoords[:, 1]
        nids_mesh = np.arange(nx*ny).reshape(nx, ny)
        n1s = nids_mesh[:-1, :-1].flatten()
        n2s = nids_mesh[1:, :-1].flatten()
        n3s = nids_mesh[1:, 1:].flatten()
        n4s = nids_mesh[:-1, 1:].flatten()
        self.N = DOF*nx*ny
        self.elements = []
        for k, (n1, n2, n3, n4) in enumerate(zip(n1s, n2s, n3s, n4s)):
            plate = BFSCPlate2D()
            plate.n1, plate.n2, plate.n3, plate.n4 = n1, n2, n3, n4
            plate.c1 = DOF*n1
            plate.c2 = DOF*n2
            plate.c3 = DOF*n3
            plate.c4 = DOF*n4
            plate.ABD = lam.ABD
            plate.lex = a/(nx - 1)
            plate.ley = b/(ny - 1)
            plate.init_k_KC0 = k*KC0_SPARSE_SIZE
            plate.init_k_KCNL = k*KCNL_SPARSE_SIZE
            plate.init_k_KG = k*KG_SPARSE_SIZE
            plate.init_k_KCfollower = k*KCfollower_SPARSE_SIZE
            self.elements.append(plate)
        self.h = lam.h

    def origin(self, plate):
        return self.x[plate.n1], self.y[plate.n1]

    def pressure(self, plate, func):
        r"""``p[i, j]`` at ``xi = points[i]``, ``eta = points[j]``"""
        x1, y1 = self.origin(plate)
        x = x1 + plate.lex*(self.points + 1)/2
        y = y1 + plate.ley*(self.points + 1)/2
        p = func(x[:, None], y[None, :])*np.ones((x.size, y.size))
        return np.ascontiguousarray(p, dtype=DOUBLE)

    def fext_follower(self, u, func, quadratic, reference=1):
        f = np.zeros(self.N, dtype=DOUBLE)
        for plate in self.elements:
            update_fext_follower(u, plate, self.points, self.weights,
                                 self.pressure(plate, func), f, quadratic,
                                 reference)
        return f

    def _matrix(self, size, call):
        n = size*len(self.elements)
        r = np.zeros(n, dtype=INT)
        c = np.zeros(n, dtype=INT)
        v = np.zeros(n, dtype=DOUBLE)
        for plate in self.elements:
            call(plate, r, c, v)
        return coo_matrix((v, (r, c)), shape=(self.N, self.N)).tocsc()

    def KCfollower(self, u, func, quadratic):
        return self._matrix(KCfollower_SPARSE_SIZE,
            lambda plate, r, c, v: update_KCfollower(u, plate, self.points,
                self.weights, self.pressure(plate, func), r, c, v,
                quadratic))

    def KC0(self):
        return self._matrix(KC0_SPARSE_SIZE,
            lambda plate, r, c, v: update_KC0(plate, self.points,
                self.weights, r, c, v))

    def KT(self, u):
        KCNL = self._matrix(KCNL_SPARSE_SIZE,
            lambda plate, r, c, v: update_KCNL(u, plate, self.points,
                self.weights, r, c, v))
        KG = self._matrix(KG_SPARSE_SIZE,
            lambda plate, r, c, v: update_KG(u, plate, self.points,
                self.weights, r, c, v))
        return self.KC0() + KCNL + KG

    def fint(self, u):
        f = np.zeros(self.N, dtype=DOUBLE)
        for plate in self.elements:
            update_fint(u, plate, self.points, self.weights, f)
        return f

    def random_state(self, rng, scale=1.):
        """In-plane displacements of the order of the thickness, deflections
        well above it, such that the nonlinear terms carry weight"""
        u = scale*self.h*rng.standard_normal(self.N)
        u[6::DOF] *= 20
        lmin = min(self.a, self.b)
        for i in (1, 2, 4, 5, 7, 8):
            u[i::DOF] *= 1/lmin
        u[7::DOF] *= 20
        u[8::DOF] *= 20
        u[9::DOF] *= 20/lmin**2
        return u

    def positions(self, plate):
        return np.concatenate([np.arange(c, c + DOF) for c in
                               (plate.c1, plate.c2, plate.c3, plate.c4)])


def boundary_flags(m, edges):
    r"""Constrained degrees of freedom: for each edge in ``edges``, a dict
    ``{'x1': 'w', ...}`` of the displacements set to zero along it, with
    their derivatives along the edge"""
    bk = np.zeros(m.N, dtype=bool)
    on = {'x1': isclose(m.x, 0), 'x2': isclose(m.x, m.a),
          'y1': isclose(m.y, 0), 'y2': isclose(m.y, m.b)}
    # value and derivative along the edge: y for the edges x = const
    along = {'x': {'u': (0, 2), 'v': (3, 5), 'w': (6, 8)},
             'y': {'u': (0, 1), 'v': (3, 4), 'w': (6, 7)}}
    for edge, fields in edges.items():
        for f in fields:
            for i in along[edge[0]][f]:
                bk[i::DOF] |= on[edge]
    return bk


from composites import isotropic_plate
from structsolve import Analysis

from bfscplate2d import (update_fint, update_KCNL, update_KG,
        update_fext_follower, update_KCfollower, DOF, DOUBLE, INT,
        KCNL_SPARSE_SIZE, KG_SPARSE_SIZE, KCfollower_SPARSE_SIZE)


class FollowerCallables(object):
    r"""The callables of structsolve.Analysis for the mesh ``m`` under the
    follower pressure ``func(x, y)`` times the load factor

    They loop over the elements with the update functions and assemble, and
    work on the unknown degrees of freedom ``bu`` only, the partition of the
    free dofs of structsolve: the solution vector of the solvers is ``c =
    u[bu]``, and the force vectors and matrices are restricted to ``bu``.
    ``quadratic`` is the truncation of the area vector.
    """
    def __init__(self, m, bu, func, quadratic):
        self.m = m
        self.bu = bu
        self.quadratic = quadratic
        self.pressures = [m.pressure(plate, func) for plate in m.elements]
        self.KC0 = m.KC0()
        N = m.N
        num_elements = len(m.elements)
        self.KCNLr = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCNLc = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCNLv = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=DOUBLE)
        self.KGr = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
        self.KGc = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
        self.KGv = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=DOUBLE)
        self.KCfr = np.zeros(KCfollower_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCfc = np.zeros(KCfollower_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCfv = np.zeros(KCfollower_SPARSE_SIZE*num_elements,
                             dtype=DOUBLE)
        # the dead part of the pressure, the force vector at u = 0
        self.fp0 = np.zeros(N, dtype=DOUBLE)
        for plate, p in zip(m.elements, self.pressures):
            update_fext_follower(np.zeros(N), plate, m.points, m.weights, p,
                                 self.fp0, quadratic)

    def expand(self, c):
        """The displacements of the model from those of the unknown dofs."""
        u = np.zeros(self.m.N, dtype=DOUBLE)
        u[self.bu] = c
        return u

    def calc_fext(self, inc=1., silent=False):
        return inc*self.fp0[self.bu]

    def calc_fint(self, c, inc=1., silent=False):
        m = self.m
        u = self.expand(c)
        fint = np.zeros(m.N, dtype=DOUBLE)
        for plate, p in zip(m.elements, self.pressures):
            update_fint(u, plate, m.points, m.weights, fint)
            # minus the part of the follower pressure that depends on u
            update_fext_follower(u, plate, m.points, m.weights, -inc*p, fint,
                                 self.quadratic, 0)
        return fint[self.bu]

    def calc_kC(self, c=None, NLgeom=False, inc=1., silent=False):
        bu = self.bu
        if not NLgeom:
            return self.KC0[bu, :][:, bu]
        m = self.m
        u = self.expand(c)
        self.KCNLv *= 0
        self.KCfv *= 0
        for plate, p in zip(m.elements, self.pressures):
            update_KCNL(u, plate, m.points, m.weights, self.KCNLr,
                        self.KCNLc, self.KCNLv)
            # the load stiffness matrix of the pressure inc*p
            update_KCfollower(u, plate, m.points, m.weights, inc*p,
                              self.KCfr, self.KCfc, self.KCfv,
                              self.quadratic)
        KCNL = coo_matrix((self.KCNLv, (self.KCNLr, self.KCNLc)),
                          shape=(m.N, m.N))
        KCf = coo_matrix((self.KCfv, (self.KCfr, self.KCfc)),
                         shape=(m.N, m.N))
        return (self.KC0 + KCNL + KCf).tocsc()[bu, :][:, bu]

    def calc_kG(self, c=None, NLgeom=False, silent=False):
        m = self.m
        u = self.expand(c)
        self.KGv *= 0
        for plate in m.elements:
            update_KG(u, plate, m.points, m.weights, self.KGr, self.KGc,
                      self.KGv)
        KG = coo_matrix((self.KGv, (self.KGr, self.KGc)), shape=(m.N, m.N))
        return KG.tocsc()[self.bu, :][:, self.bu]

In [8]:
@pytest.mark.parametrize('quadratic', TRUNCATIONS)
def test_cantilever_large_deflection_against_panels(quadratic):
    r"""The square isotropic cantilever of ``test_follower_pressure.py``,
    clamped at `x = 0`, the other edges free, under a uniform follower
    pressure pushing along `+z`, in five equal load increments, against
    panels within 1e-3 for the tip deflection `w` and the tip displacement
    `u`"""
    E, nu, h = 70.e9, 0.3, 2.e-3
    a = b = 0.3
    nel = 10
    m = Mesh(nx=nel + 1, ny=nel + 1, a=a, b=b,
             lam=isotropic_plate(thickness=h, E=E, nu=nu))
    bk = boundary_flags(m, {'x1': 'uvw'})
    bk[7::DOF] |= isclose(m.x, 0)
    bk[9::DOF] |= isclose(m.x, 0)
    bu = ~bk
    func = lambda x, y: 2.e3 + 0*x*y
    tip = np.where(isclose(m.x, a) & isclose(m.y, b/2))[0][0]

    callables = FollowerCallables(m, bu, func, quadratic)
    an = Analysis(calc_fext=callables.calc_fext,
                  calc_fint=callables.calc_fint,
                  calc_kC=callables.calc_kC,
                  calc_kG=callables.calc_kG)
    an.initialInc = 0.2
    an.maxInc = 0.2
    increments, cs = an.static(NLgeom=True)
    # every increment converged, none was cut
    assert np.allclose(increments, [0.2, 0.4, 0.6, 0.8, 1.]), increments
    u = callables.expand(cs[-1])

    w_tip = u[DOF*tip + 6]/h
    u_tip = u[DOF*tip]
    print('w/h', w_tip, 'u', u_tip)
    assert isclose(w_tip, PANELS[quadratic]['w'], rtol=1.e-3)
    assert isclose(u_tip, PANELS[quadratic]['u'], rtol=1.e-3)

Running the analysis for the two truncations of the area vector, which prints the tip deflection over the thickness and the tip displacement $u$:

In [9]:
for quadratic in TRUNCATIONS:
    test_cantilever_large_deflection_against_panels(quadratic)

Started Non-Linear Static Analysis
	___________________________________________
	                                           
	Newton-Raphson solver
	___________________________________________
	Initializing...
	Step 1, attempting load factor 0.20000
		Iteration: 1, max(|R|) = 3.067e+03, zeta = 1.000e+00


		Iteration: 2, max(|R|) = 6.677e-02, zeta = 7.519e-02


		Iteration: 3, max(|R|) = 5.288e-02, zeta = 8.765e-03


		Iteration: 4, max(|R|) = 6.037e-07, zeta = 9.849e-08


		Iteration: 5, max(|R|) = 1.101e-11, zeta = 2.641e-12
		Converged at load factor 0.20000
	Step 2, attempting load factor 0.40000
		Iteration: 1, max(|R|) = 6.089e+03, zeta = 1.000e+00


		Iteration: 2, max(|R|) = 3.834e-01, zeta = 2.133e-01


... (34 outputs left out)

		Iteration: 2, max(|R|) = 1.306e-01, zeta = 3.192e-02


		Iteration: 3, max(|R|) = 2.058e-01, zeta = 1.114e-02


		Iteration: 4, max(|R|) = 1.891e-08, zeta = 5.565e-09


		Iteration: 5, max(|R|) = 6.614e-10, zeta = 1.727e-11
		Converged at load factor 1.00000
Finished Non-Linear Static Analysis
	    load factor 1.00000
w/h 20.226083352268986 u -0.0031229503463031066


Both truncations of the area vector give the tip deflection
$w/h = 20.23$ and the tip displacement $u = -3.12 \times 10^{-3}$ m.

## References

1. Hibbitt, H. D., Some follower forces and load stiffness, *International Journal for Numerical Methods in Engineering*, 14(6), 937-941, 1979, doi:10.1002/nme.1620140613.
2. Schweizerhof, K., and Ramm, E., Displacement dependent pressure loads in nonlinear finite element analyses, *Computers & Structures*, 18(6), 1099-1114, 1984, doi:10.1016/0045-7949(84)90154-8.
3. Castro, S. G. P., and Donadon, M. V., Assembly of semi-analytical models to address linear buckling and vibration of stiffened composite panels with debonding defect, *Composite Structures*, 160, 232-247, 2017, doi:10.1016/j.compstruct.2016.10.026.